# LSSO CV A100 Colab Experiments

This notebook writes and runs the latest clean LSSO CV script.

Main table plan:
- CIFAR-10 / CIFAR-100, `patch=2`
- ImageNet-100, `patch=8`
- MHA, LSSO-r16, LSSO-r32
- Optional elastic rank inference: train r32, evaluate r24/r16/r8

LSSO core: U RMS norm + positive mu + bounded gamma + fp32 small SPD solve; no operator-level dropout.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

Sun May 31 03:58:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip -q install kaggle tqdm

## Optional ImageNet-100 download

For ImageNet-100, upload `kaggle.json`, then run this cell. If the Kaggle mirror differs, edit `KAGGLE_DATASET`.
The script expects an ImageFolder structure containing `train/` and `val/`.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("ambityga/imagenet100")
print("ImageNet-100 downloaded to:", path)

100%|██████████| 16.1G/16.1G [01:32<00:00, 187MB/s]

Extracting files...


ImageNet-100 downloaded to: /root/.cache/kagglehub/datasets/ambityga/imagenet100/versions/8


## Write training script

In [ ]:
%%writefile /content/lsso_cv_colab.py
from __future__ import annotations

import argparse, csv, json, math, os, random, time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm


def set_seed(seed:int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def amp_dtype(name:str):
    return {"none":None,"bf16":torch.bfloat16,"fp16":torch.float16}[name]


def topk_acc(logits, y, topk=(1,5)):
    maxk=min(max(topk), logits.shape[1]); _, pred=logits.topk(maxk,1); pred=pred.t(); corr=pred.eq(y.view(1,-1))
    out=[]
    for k in topk:
        k=min(k, logits.shape[1]); out.append(corr[:k].reshape(-1).float().sum().item()*100.0/y.numel())
    return out

class Meter:
    def __init__(self): self.s=0.0; self.n=0
    def update(self, v, n=1): self.s += float(v)*int(n); self.n += int(n)
    @property
    def avg(self): return self.s/max(1,self.n)

# ---------------- Mixup/CutMix ----------------
def rand_bbox(size, lam):
    H,W=size[2],size[3]; cut_rat=math.sqrt(1-lam); cw,ch=int(W*cut_rat),int(H*cut_rat)
    cx,cy=np.random.randint(W),np.random.randint(H)
    return np.clip(cx-cw//2,0,W),np.clip(cy-ch//2,0,H),np.clip(cx+cw//2,0,W),np.clip(cy+ch//2,0,H)

def mix_batch(x,y,mixup=0.0,cutmix=0.0):
    if mixup<=0 and cutmix<=0: return x,y,None,1.0
    use_cut = cutmix>0 and (mixup<=0 or random.random()<0.5); alpha=cutmix if use_cut else mixup
    lam=float(np.random.beta(alpha,alpha)); idx=torch.randperm(x.size(0),device=x.device); ya,yb=y,y[idx]
    if use_cut:
        x1,y1,x2,y2=rand_bbox(x.size(),lam); x[:,:,y1:y2,x1:x2]=x[idx,:,y1:y2,x1:x2]
        lam=1-((x2-x1)*(y2-y1)/(x.size(-1)*x.size(-2)))
    else:
        x=lam*x+(1-lam)*x[idx]
    return x,ya,yb,lam

def mixed_ce(crit, logits, ya, yb, lam):
    return crit(logits,ya) if yb is None else lam*crit(logits,ya)+(1-lam)*crit(logits,yb)

class DropPath(nn.Module):
    def __init__(self,p=0.0): super().__init__(); self.p=float(p)
    def forward(self,x):
        if self.p==0 or not self.training: return x
        keep=1-self.p; shape=(x.shape[0],)+(1,)*(x.ndim-1); r=keep+torch.rand(shape,device=x.device,dtype=x.dtype); r.floor_(); return x/keep*r

# ---------------- LSSO ----------------
@dataclass
class LSSODiag:
    gamma_over_mu: float
    correction_ratio: float
    effective_rank: float


def lsso_core(U,C,mu,gamma,eye,return_aux=False):
    B,H,N,r=U.shape; dh=C.shape[-1]
    mu=mu.view(1,H,1,1); gamma=gamma.view(1,H,1,1); inv_mu=mu.reciprocal(); local=inv_mu*C
    Uh=U.flatten(0,1); Ch=C.flatten(0,1); Ut=Uh.transpose(1,2)
    UtU=torch.bmm(Ut,Uh).view(B,H,r,r); UtC=torch.bmm(Ut,Ch).view(B,H,r,dh)
    G=eye[:,:,:r,:r].float() + (gamma*inv_mu).float()*UtU.float()
    rhs=UtC.float()
    try:
        L=torch.linalg.cholesky(G.view(B*H,r,r)); K=torch.cholesky_solve(rhs.view(B*H,r,dh),L)
    except RuntimeError:
        K=torch.linalg.solve_ex(G.view(B*H,r,r), rhs.view(B*H,r,dh), check_errors=False).result
    K=K.to(U.dtype).view(B,H,r,dh)
    UK=torch.bmm(Uh, K.view(B*H,r,dh)).view(B,H,N,dh)
    corr=gamma*inv_mu*inv_mu*UK; Y=local-corr
    return (Y,local,corr,UtU) if return_aux else Y

class LSSO(nn.Module):
    def __init__(self,dim,heads,rank,gamma_max=0.3,theta_gamma_init=-4.0,eps=1e-5,no_global=False):
        super().__init__(); assert dim%heads==0
        self.dim=dim; self.heads=heads; self.rank=rank; self.active_rank=rank; self.dh=dim//heads; self.eps=eps; self.gamma_max=gamma_max; self.no_global=no_global
        self.w_uc=nn.Linear(dim,heads*rank+dim,bias=False); self.w_o=nn.Linear(dim,dim,bias=False)
        self.theta_mu=nn.Parameter(torch.zeros(heads)); self.theta_gamma=nn.Parameter(torch.full((heads,),float(theta_gamma_init)))
        self.register_buffer('eye',torch.eye(rank).view(1,1,rank,rank),persistent=False)
        self.record_diagnostics=False; self.last_diag:Optional[LSSODiag]=None
    def set_active_rank(self,r=None):
        self.active_rank=self.rank if r is None else int(r)
        if self.active_rank<1 or self.active_rank>self.rank: raise ValueError('bad active rank')
    def forward(self,x):
        B,N,D=x.shape; H=self.heads; r=self.active_rank; dh=self.dh
        UC=self.w_uc(x); Uall,C=UC.split((H*self.rank,D),dim=-1)
        U=Uall.view(B,N,H,self.rank).transpose(1,2).contiguous()[:,:,:,:r].contiguous()
        C=C.view(B,N,H,dh).transpose(1,2).contiguous()
        U=U*torch.rsqrt(torch.mean(U*U,dim=-1,keepdim=True)+self.eps)  # essential U RMS norm
        mu=F.softplus(self.theta_mu)+self.eps; gamma=self.gamma_max*torch.sigmoid(self.theta_gamma)
        if self.no_global: gamma=torch.zeros_like(gamma)
        if self.record_diagnostics:
            Y,local,corr,UtU=lsso_core(U,C,mu,gamma,self.eye,True)
            with torch.no_grad():
                eig=torch.linalg.eigvalsh(UtU.float()).clamp_min(0); s=eig.sum(-1)
                er=(s*s)/(eig.square().sum(-1).clamp_min(self.eps)); ratio=corr.float().norm(dim=(-2,-1))/(local.float().norm(dim=(-2,-1)).clamp_min(self.eps))
                self.last_diag=LSSODiag(float((gamma/mu).mean().cpu()),float(ratio.mean().cpu()),float(er.mean().cpu()))
        else:
            Y=lsso_core(U,C,mu,gamma,self.eye,False)
        return self.w_o(Y.transpose(1,2).contiguous().view(B,N,D))

# ---------------- Model ----------------
class PatchEmbed(nn.Module):
    def __init__(self,img,patch,dim):
        super().__init__(); assert img%patch==0; self.num_patches=(img//patch)**2; self.proj=nn.Conv2d(3,dim,patch,patch)
    def forward(self,x): return self.proj(x).flatten(2).transpose(1,2)

class MLP(nn.Module):
    def __init__(self,dim,ratio=4.0,drop=0.0):
        super().__init__(); h=int(dim*ratio); self.net=nn.Sequential(nn.Linear(dim,h),nn.GELU(),nn.Dropout(drop),nn.Linear(h,dim),nn.Dropout(drop))
    def forward(self,x): return self.net(x)

class Block(nn.Module):
    def __init__(self,dim,heads,mixer,rank,mlp_ratio,drop,drop_path,gamma_max,theta_gamma_init):
        super().__init__(); self.n1=nn.LayerNorm(dim); self.is_mha=(mixer=='mha')
        if self.is_mha: self.mix=nn.MultiheadAttention(dim,heads,dropout=drop,batch_first=True)
        else: self.mix=LSSO(dim,heads,rank,gamma_max,theta_gamma_init,no_global=(mixer=='lsso-no-global'))
        self.dp=DropPath(drop_path); self.n2=nn.LayerNorm(dim); self.mlp=MLP(dim,mlp_ratio,drop)
    def forward(self,x):
        z=self.n1(x); z=self.mix(z,z,z,need_weights=False)[0] if self.is_mha else self.mix(z)
        x=x+self.dp(z); x=x+self.dp(self.mlp(self.n2(x))); return x

class VisionMixer(nn.Module):
    def __init__(self,img,patch,classes,dim,depth,heads,mixer,rank,mlp_ratio=4,drop=0,drop_path=0,gamma_max=0.3,theta_gamma_init=-4):
        super().__init__(); self.patch=PatchEmbed(img,patch,dim); self.pos=nn.Parameter(torch.zeros(1,self.patch.num_patches,dim))
        dpr=torch.linspace(0,drop_path,depth).tolist()
        self.blocks=nn.ModuleList([Block(dim,heads,mixer,rank,mlp_ratio,drop,dpr[i],gamma_max,theta_gamma_init) for i in range(depth)])
        self.norm=nn.LayerNorm(dim); self.head=nn.Linear(dim,classes); self.reset_parameters()
    def reset_parameters(self):
        nn.init.trunc_normal_(self.pos,std=0.02)
        for m in self.modules():
            if isinstance(m,(nn.Linear,nn.Conv2d)): nn.init.trunc_normal_(m.weight,std=0.02); (nn.init.zeros_(m.bias) if getattr(m,'bias',None) is not None else None)
            elif isinstance(m,nn.LayerNorm): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def set_active_rank(self,r=None):
        for m in self.modules():
            if isinstance(m,LSSO): m.set_active_rank(r)
    def set_diag(self,on):
        for m in self.modules():
            if isinstance(m,LSSO): m.record_diagnostics=on
    def diag(self):
        vals=[m.last_diag for m in self.modules() if isinstance(m,LSSO) and m.last_diag]
        return {} if not vals else {'diag_gamma_over_mu':float(np.mean([v.gamma_over_mu for v in vals])),'diag_correction_ratio':float(np.mean([v.correction_ratio for v in vals])),'diag_effective_rank':float(np.mean([v.effective_rank for v in vals]))}
    def forward(self,x):
        x=self.patch(x)+self.pos
        for b in self.blocks: x=b(x)
        return self.head(self.norm(x).mean(1))

# ---------------- Data ----------------
def transforms_for(ds,img,ra):
    if ds.startswith('cifar'):
        mean=(0.4914,0.4822,0.4465) if ds=='cifar10' else (0.5071,0.4867,0.4408); std=(0.247,0.2435,0.2616) if ds=='cifar10' else (0.2675,0.2565,0.2761)
        tr=[transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip()]
        if ra: tr.append(transforms.RandAugment(2,9))
        return transforms.Compose(tr+[transforms.ToTensor(),transforms.Normalize(mean,std)]), transforms.Compose([transforms.ToTensor(),transforms.Normalize(mean,std)])
    mean=(0.485,0.456,0.406); std=(0.229,0.224,0.225)
    tr=[transforms.RandomResizedCrop(img,interpolation=transforms.InterpolationMode.BICUBIC),transforms.RandomHorizontalFlip()]
    if ra: tr.append(transforms.RandAugment(2,9))
    va=[transforms.Resize(int(img*256/224),interpolation=transforms.InterpolationMode.BICUBIC),transforms.CenterCrop(img)]
    return transforms.Compose(tr+[transforms.ToTensor(),transforms.Normalize(mean,std)]), transforms.Compose(va+[transforms.ToTensor(),transforms.Normalize(mean,std)])

def find_splits(root):
    root=Path(root); bases=[root,root/'imagenet100',root/'ImageNet100',root/'data']
    for b in bases:
        for tr in ['train','training']:
            for va in ['val','valid','validation','test']:
                if (b/tr).exists() and (b/va).exists(): return b/tr,b/va
    raise FileNotFoundError(f'No train/val ImageFolder split under {root}')

def build_data(args):
    trf,vaf=transforms_for(args.dataset,args.image_size,args.randaugment)
    if args.dataset=='cifar10': return datasets.CIFAR10(args.data_root,True,trf,download=True),datasets.CIFAR10(args.data_root,False,vaf,download=True),10
    if args.dataset=='cifar100': return datasets.CIFAR100(args.data_root,True,trf,download=True),datasets.CIFAR100(args.data_root,False,vaf,download=True),100
    tr,va=find_splits(args.data_root); train=datasets.ImageFolder(tr,trf); val=datasets.ImageFolder(va,vaf); return train,val,len(train.classes)

def macs(mixer,N,D,H,r):
    if mixer=='mha': return 4*N*D*D + 2*N*N*D
    dh=D//H; return N*D*(H*r+D)+N*D*D+H*(N*r*r+2*N*r*dh+r**3+r*r*dh)

@torch.no_grad()
def evaluate(model,loader,crit,device,dtype=None,active_rank=None,desc='eval'):
    model.eval(); model.set_active_rank(active_rank); model.set_diag(True); lm=Meter(); t1=Meter(); t5=Meter(); n=0; t0=time.time()
    for x,y in tqdm(loader,desc=desc,leave=False):
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        with torch.amp.autocast('cuda',dtype=dtype,enabled=(dtype is not None and device.type=='cuda')):
            out=model(x); loss=crit(out,y)
        a1,a5=topk_acc(out.float(),y); bs=x.size(0); lm.update(loss.item(),bs); t1.update(a1,bs); t5.update(a5,bs); n+=bs
    d=model.diag(); model.set_diag(False)
    return {'loss':lm.avg,'top1':t1.avg,'top5':t5.avg,'samples_per_sec':n/max(1e-9,time.time()-t0),**d}

def train_epoch(model,loader,opt,crit,scaler,device,args,dtype,epoch,sched=None):
    model.train(); model.set_active_rank(None); lm=Meter(); t1=Meter(); n=0; t0=time.time(); opt.zero_grad(set_to_none=True)
    pbar=tqdm(loader,desc=f'train {epoch}',leave=False)
    for i,(x,y) in enumerate(pbar,1):
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True); x,ya,yb,lam=mix_batch(x,y,args.mixup,args.cutmix)
        with torch.amp.autocast('cuda',dtype=dtype,enabled=(dtype is not None and device.type=='cuda')):
            out=model(x); loss=mixed_ce(crit,out,ya,yb,lam)
        if scaler is not None:
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),args.grad_clip); scaler.step(opt); scaler.update()
        else:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),args.grad_clip); opt.step()
        if sched is not None: sched.step()
        opt.zero_grad(set_to_none=True); a1,_=topk_acc(out.detach().float(),y); bs=x.size(0); lm.update(loss.item(),bs); t1.update(a1,bs); n+=bs
        if i%args.log_every==0: pbar.set_postfix(loss=f'{lm.avg:.4f}',top1=f'{t1.avg:.2f}')
    return {'loss':lm.avg,'top1':t1.avg,'samples_per_sec':n/max(1e-9,time.time()-t0)}

def scheduler(opt,epochs,steps,warmup_epochs):
    total=epochs*steps; warm=warmup_epochs*steps
    def f(s):
        if warm>0 and s<warm: return (s+1)/warm
        p=(s-warm)/max(1,total-warm); return 0.5*(1+math.cos(math.pi*p))
    return torch.optim.lr_scheduler.LambdaLR(opt,f)

def append_csv(path,row):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); exists=path.exists(); keys=list(row.keys())
    with path.open('a',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=keys); (w.writeheader() if not exists else None); w.writerow(row)

def ranks(s): return [] if not s.strip() else [int(x) for x in s.split(',') if x.strip()]

def main():
    p=argparse.ArgumentParser()
    p.add_argument('--dataset',choices=['cifar10','cifar100','imagefolder'],required=True); p.add_argument('--data-root',default='/content/data'); p.add_argument('--out-dir',default='/content/lsso_cv_runs')
    p.add_argument('--mixer',choices=['mha','lsso','lsso-no-global'],default='lsso'); p.add_argument('--rank',type=int,default=32)
    p.add_argument('--image-size',type=int,default=32); p.add_argument('--patch-size',type=int,default=2); p.add_argument('--dim',type=int,default=96); p.add_argument('--depth',type=int,default=3); p.add_argument('--heads',type=int,default=6)
    p.add_argument('--mlp-ratio',type=float,default=4); p.add_argument('--dropout',type=float,default=0); p.add_argument('--drop-path',type=float,default=0); p.add_argument('--gamma-max',type=float,default=0.3); p.add_argument('--theta-gamma-init',type=float,default=-4)
    p.add_argument('--epochs',type=int,default=60); p.add_argument('--batch-size',type=int,default=256); p.add_argument('--workers',type=int,default=4); p.add_argument('--lr',type=float,default=5e-4); p.add_argument('--weight-decay',type=float,default=0.05); p.add_argument('--warmup-epochs',type=int,default=5)
    p.add_argument('--label-smoothing',type=float,default=0.1); p.add_argument('--mixup',type=float,default=0); p.add_argument('--cutmix',type=float,default=0); p.add_argument('--randaugment',action='store_true'); p.add_argument('--grad-clip',type=float,default=1.0); p.add_argument('--amp',choices=['bf16','fp16','none'],default='bf16')
    p.add_argument('--seed',type=int,default=1); p.add_argument('--log-every',type=int,default=50); p.add_argument('--elastic-eval-ranks',default=''); p.add_argument('--save-best',action=argparse.BooleanOptionalAction,default=True)
    args=p.parse_args(); set_seed(args.seed); torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True; torch.backends.cudnn.benchmark=True
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); dtype=amp_dtype(args.amp); dtype=dtype if device.type=='cuda' else None
    print(json.dumps({'event':'start','torch':torch.__version__,'cuda':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'args':vars(args)},ensure_ascii=False),flush=True)
    train,val,classes=build_data(args); train_loader=DataLoader(train,batch_size=args.batch_size,shuffle=True,num_workers=args.workers,pin_memory=device.type=='cuda',persistent_workers=args.workers>0,drop_last=True); val_loader=DataLoader(val,batch_size=args.batch_size,shuffle=False,num_workers=args.workers,pin_memory=device.type=='cuda',persistent_workers=args.workers>0)
    model=VisionMixer(args.image_size,args.patch_size,classes,args.dim,args.depth,args.heads,args.mixer,args.rank,args.mlp_ratio,args.dropout,args.drop_path,args.gamma_max,args.theta_gamma_init).to(device)
    params=sum(p.numel() for p in model.parameters()); N=(args.image_size//args.patch_size)**2; est_macs=macs(args.mixer,N,args.dim,args.heads,args.rank)
    crit=nn.CrossEntropyLoss(label_smoothing=args.label_smoothing); opt=torch.optim.AdamW(model.parameters(),lr=args.lr,weight_decay=args.weight_decay); sched=scheduler(opt,args.epochs,len(train_loader),args.warmup_epochs); scaler=torch.amp.GradScaler('cuda',enabled=(dtype==torch.float16 and device.type=='cuda'))
    out=Path(args.out_dir); out.mkdir(parents=True,exist_ok=True); run=f'{args.dataset}_{args.mixer}_r{args.rank}_d{args.dim}_L{args.depth}_h{args.heads}_img{args.image_size}_p{args.patch_size}_s{args.seed}'; log=out/f'{run}.jsonl'; ckpt=out/f'{run}_best.pt'; csvp=out/'summary.csv'
    header={'event':'header','run':run,'params':params,'tokens':N,'mixer_macs_est':est_macs,'classes':classes,'train_size':len(train),'val_size':len(val)}; print(json.dumps(header),flush=True); log.write_text(json.dumps({'args':vars(args),**header})+'\n')
    best=-1
    for ep in range(1,args.epochs+1):
        tr=train_epoch(model,train_loader,opt,crit,scaler,device,args,dtype,ep,sched); va=evaluate(model,val_loader,crit,device,dtype,None,'val')
        row={'event':'epoch','epoch':ep,'lr':opt.param_groups[0]['lr'],'train_loss':tr['loss'],'train_top1':tr['top1'],'val_loss':va['loss'],'val_top1':va['top1'],'val_top5':va['top5'],'train_samples_per_sec':tr['samples_per_sec'],'val_samples_per_sec':va['samples_per_sec'],'max_mem_gb':torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0,**{k:v for k,v in va.items() if k.startswith('diag_')}}
        print(json.dumps(row),flush=True); log.open('a').write(json.dumps(row)+'\n')
        if va['top1']>best:
            best=va['top1'];
            if args.save_best: torch.save({'model':model.state_dict(),'args':vars(args),'best_top1':best,'epoch':ep},ckpt)
    if args.save_best and ckpt.exists(): model.load_state_dict(torch.load(ckpt,map_location=device)['model'])
    final=evaluate(model,val_loader,crit,device,dtype,None,'final')
    base={'dataset':args.dataset,'mixer':args.mixer,'rank':args.rank,'active_rank':args.rank if args.mixer!='mha' else 0,'dim':args.dim,'depth':args.depth,'heads':args.heads,'image_size':args.image_size,'patch_size':args.patch_size,'tokens':N,'seed':args.seed,'params':params,'mixer_macs_est':est_macs,'best_top1':best,'final_top1':final['top1'],'final_top5':final['top5'],'final_loss':final['loss'],'final_samples_per_sec':final['samples_per_sec'],'max_mem_gb':torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0,**{k:final.get(k,'') for k in ['diag_gamma_over_mu','diag_correction_ratio','diag_effective_rank']}}
    append_csv(csvp,base); print(json.dumps({'event':'final',**base}),flush=True)
    if args.mixer!='mha':
        for ar in ranks(args.elastic_eval_ranks):
            if ar>args.rank: continue
            m=evaluate(model,val_loader,crit,device,dtype,ar,f'infer-r{ar}'); row={**base,'active_rank':ar,'mixer_macs_est':macs(args.mixer,N,args.dim,args.heads,ar),'final_top1':m['top1'],'final_top5':m['top5'],'final_loss':m['loss'],'final_samples_per_sec':m['samples_per_sec'],**{k:m.get(k,'') for k in ['diag_gamma_over_mu','diag_correction_ratio','diag_effective_rank']}}
            append_csv(csvp,row); print(json.dumps({'event':'elastic_eval',**row}),flush=True)
    print('logs:',log); print('summary:',csvp); print('ckpt:',ckpt if args.save_best else 'disabled')

if __name__=='__main__': main()


Overwriting /content/lsso_cv_colab.py


## Smoke test

In [ ]:
!python /content/lsso_cv_colab.py \
  --dataset cifar100 --data-root /content/data --out-dir /content/lsso_cv_runs \
  --mixer lsso --rank 32 \
  --image-size 32 --patch-size 2 --dim 96 --depth 3 --heads 6 \
  --epochs 1 --batch-size 256 --lr 5e-4 --warmup-epochs 1 \
  --amp bf16 --elastic-eval-ranks 32,16,8

{"event": "start", "torch": "2.11.0+cu128", "cuda": true, "gpu": "NVIDIA A100-SXM4-40GB", "args": {"dataset": "cifar100", "data_root": "/content/data", "out_dir": "/content/lsso_cv_runs", "mixer": "lsso", "rank": 32, "image_size": 32, "patch_size": 2, "dim": 96, "depth": 3, "heads": 6, "mlp_ratio": 4, "dropout": 0, "drop_path": 0, "gamma_max": 0.3, "theta_gamma_init": -4, "epochs": 1, "batch_size": 256, "workers": 4, "lr": 0.0005, "weight_decay": 0.05, "warmup_epochs": 1, "label_smoothing": 0.1, "mixup": 0, "cutmix": 0, "randaugment": false, "grad_clip": 1.0, "amp": "bf16", "seed": 1, "log_every": 50, "elastic_eval_ranks": "32,16,8", "save_best": true}}
100% 169M/169M [00:01<00:00, 92.9MB/s]
{"event": "header", "run": "cifar100_lsso_r32_d96_L3_h6_img32_p2_s1", "params": 370120, "tokens": 256, "mixer_macs_est": 12877824, "classes": 100, "train_size": 50000, "val_size": 10000}
{"event": "epoch", "epoch": 1, "lr": 0.0005, "train_loss": 4.404656588725555, "train_top1": 4.609375, "val_loss"

## CIFAR-100 main runs, seed=1. Repeat with `--seed 2` and `--seed 3`.

In [ ]:
# CIFAR-100 MHA
!python /content/lsso_cv_colab.py \
  --dataset cifar100 --data-root /content/data --out-dir /content/lsso_cv_runs \
  --mixer mha --rank 32 \
  --image-size 32 --patch-size 2 --dim 96 --depth 3 --heads 6 \
  --epochs 100 --batch-size 256 --lr 5e-4 --warmup-epochs 5 \
  --label-smoothing 0.1 --amp bf16 --seed 1

{"event": "start", "torch": "2.11.0+cu128", "cuda": true, "gpu": "NVIDIA A100-SXM4-40GB", "args": {"dataset": "cifar100", "data_root": "/content/data", "out_dir": "/content/lsso_cv_runs", "mixer": "mha", "rank": 32, "image_size": 32, "patch_size": 2, "dim": 96, "depth": 3, "heads": 6, "mlp_ratio": 4, "dropout": 0, "drop_path": 0, "gamma_max": 0.3, "theta_gamma_init": -4, "epochs": 100, "batch_size": 256, "workers": 4, "lr": 0.0005, "weight_decay": 0.05, "warmup_epochs": 5, "label_smoothing": 0.1, "mixup": 0, "cutmix": 0, "randaugment": false, "grad_clip": 1.0, "amp": "bf16", "seed": 1, "log_every": 50, "elastic_eval_ranks": "", "save_best": true}}
{"event": "header", "run": "cifar100_mha_r32_d96_L3_h6_img32_p2_s1", "params": 371236, "tokens": 256, "mixer_macs_est": 22020096, "classes": 100, "train_size": 50000, "val_size": 10000}
{"event": "epoch", "epoch": 1, "lr": 0.00010051282051282052, "train_loss": 4.504680628654285, "train_top1": 3.5576923076923075, "val_loss": 4.386607695007324,

In [ ]:
# CIFAR-100 LSSO-r16
!python /content/lsso_cv_colab.py \
  --dataset cifar100 --data-root /content/data --out-dir /content/lsso_cv_runs \
  --mixer lsso --rank 16 \
  --image-size 32 --patch-size 2 --dim 96 --depth 3 --heads 6 \
  --epochs 100 --batch-size 256 --lr 5e-4 --warmup-epochs 5 \
  --label-smoothing 0.1 --amp bf16 --seed 1

{"event": "start", "torch": "2.11.0+cu128", "cuda": true, "gpu": "NVIDIA A100-SXM4-40GB", "args": {"dataset": "cifar100", "data_root": "/content/data", "out_dir": "/content/lsso_cv_runs", "mixer": "lsso", "rank": 16, "image_size": 32, "patch_size": 2, "dim": 96, "depth": 3, "heads": 6, "mlp_ratio": 4, "dropout": 0, "drop_path": 0, "gamma_max": 0.3, "theta_gamma_init": -4, "epochs": 100, "batch_size": 256, "workers": 4, "lr": 0.0005, "weight_decay": 0.05, "warmup_epochs": 5, "label_smoothing": 0.1, "mixup": 0, "cutmix": 0, "randaugment": false, "grad_clip": 1.0, "amp": "bf16", "seed": 1, "log_every": 50, "elastic_eval_ranks": "", "save_best": true}}
{"event": "header", "run": "cifar100_lsso_r16_d96_L3_h6_img32_p2_s1", "params": 342472, "tokens": 256, "mixer_macs_est": 8306688, "classes": 100, "train_size": 50000, "val_size": 10000}
{"event": "epoch", "epoch": 1, "lr": 0.00010051282051282052, "train_loss": 4.5356376599042845, "train_top1": 3.1189903846153846, "val_loss": 4.41991605300903

In [ ]:
# CIFAR-100 LSSO-r32 + elastic rank inference
!python /content/lsso_cv_colab.py \
  --dataset cifar100 --data-root /content/data --out-dir /content/lsso_cv_runs \
  --mixer lsso --rank 32 \
  --image-size 32 --patch-size 2 --dim 96 --depth 3 --heads 6 \
  --epochs 100 --batch-size 256 --lr 5e-4 --warmup-epochs 5 \
  --label-smoothing 0.1 --amp bf16 --seed 1 \

{"event": "start", "torch": "2.11.0+cu128", "cuda": true, "gpu": "NVIDIA A100-SXM4-40GB", "args": {"dataset": "cifar100", "data_root": "/content/data", "out_dir": "/content/lsso_cv_runs", "mixer": "lsso", "rank": 32, "image_size": 32, "patch_size": 2, "dim": 96, "depth": 3, "heads": 6, "mlp_ratio": 4, "dropout": 0, "drop_path": 0, "gamma_max": 0.3, "theta_gamma_init": -4, "epochs": 100, "batch_size": 256, "workers": 4, "lr": 0.0005, "weight_decay": 0.05, "warmup_epochs": 5, "label_smoothing": 0.1, "mixup": 0, "cutmix": 0, "randaugment": false, "grad_clip": 1.0, "amp": "bf16", "seed": 1, "log_every": 50, "elastic_eval_ranks": "32,24,16,8", "save_best": true}}
{"event": "header", "run": "cifar100_lsso_r32_d96_L3_h6_img32_p2_s1", "params": 370120, "tokens": 256, "mixer_macs_est": 12877824, "classes": 100, "train_size": 50000, "val_size": 10000}
{"event": "epoch", "epoch": 1, "lr": 0.00010051282051282052, "train_loss": 4.534253367399558, "train_top1": 3.1630608974358974, "val_loss": 4.4180

For CIFAR-10, change `--dataset cifar100` to `--dataset cifar10`.

## ImageNet-100 patch=8 main runs

In [ ]:
import os
import kagglehub
from pathlib import Path

raw_root = kagglehub.dataset_download("ambityga/imagenet100")
print("Raw downloaded path:", raw_root)

# Create a unified directory structure using symlinks for PyTorch ImageFolder
IMAGENET100_ROOT = "/content/data/imagenet100"
dst_root = Path(IMAGENET100_ROOT)
(dst_root / 'train').mkdir(parents=True, exist_ok=True)
(dst_root / 'val').mkdir(parents=True, exist_ok=True)

raw_path = Path(raw_root)

# Find and symlink train folders (handles train.X1, train.X2, etc.)
for d in raw_path.glob('*train*'):
    if d.is_dir():
        for cls_dir in d.iterdir():
            if cls_dir.is_dir():
                dst = dst_root / 'train' / cls_dir.name
                if not dst.exists():
                    os.symlink(cls_dir, dst)

# Find and symlink val folders
for d in raw_path.glob('*val*'):
    if d.is_dir():
        for cls_dir in d.iterdir():
            if cls_dir.is_dir():
                dst = dst_root / 'val' / cls_dir.name
                if not dst.exists():
                    os.symlink(cls_dir, dst)

print(f"Unified dataset created at: {IMAGENET100_ROOT}")
print(f"Train classes found: {len(list((dst_root / 'train').iterdir()))}")
print(f"Val classes found: {len(list((dst_root / 'val').iterdir()))}")

100%|██████████| 16.1G/16.1G [01:49<00:00, 158MB/s]

Extracting files...


Raw downloaded path: /root/.cache/kagglehub/datasets/ambityga/imagenet100/versions/8
Unified dataset created at: /content/data/imagenet100
Train classes found: 100
Val classes found: 100


In [ ]:
# ImageNet-100 MHA patch=8
!python /content/lsso_cv_colab.py \
  --dataset imagefolder --data-root "{IMAGENET100_ROOT}" --out-dir /content/lsso_cv_runs \
  --mixer mha --rank 32 \
  --image-size 224 --patch-size 8 --dim 256 --depth 8 --heads 8 \
  --epochs 120 --batch-size 256 --workers 12 --lr 1e-3 --warmup-epochs 10 \
  --weight-decay 0.05 --label-smoothing 0.1 --randaugment --mixup 0.8 --cutmix 1.0 \
  --drop-path 0.1 --amp bf16 --seed 1

{"event": "start", "torch": "2.11.0+cu128", "cuda": true, "gpu": "NVIDIA A100-SXM4-40GB", "args": {"dataset": "imagefolder", "data_root": "/content/data/imagenet100", "out_dir": "/content/lsso_cv_runs", "mixer": "mha", "rank": 32, "image_size": 224, "patch_size": 8, "dim": 256, "depth": 8, "heads": 8, "mlp_ratio": 4, "dropout": 0, "drop_path": 0.1, "gamma_max": 0.3, "theta_gamma_init": -4, "epochs": 120, "batch_size": 256, "workers": 12, "lr": 0.001, "weight_decay": 0.05, "warmup_epochs": 10, "label_smoothing": 0.1, "mixup": 0.8, "cutmix": 1.0, "randaugment": true, "grad_clip": 1.0, "amp": "bf16", "seed": 1, "log_every": 50, "elastic_eval_ranks": "", "save_best": true}}
{"event": "header", "run": "imagefolder_mha_r32_d256_L8_h8_img224_p8_s1", "params": 6594404, "tokens": 784, "mixer_macs_est": 520224768, "classes": 100, "train_size": 130000, "val_size": 5000}
{"event": "epoch", "epoch": 1, "lr": 0.00010019723865877712, "train_loss": 4.477598725456223, "train_top1": 3.081083579881657, "

In [ ]:
# ImageNet-100 LSSO-r16 patch=8
!python /content/lsso_cv_colab.py \
  --dataset imagefolder --data-root "{IMAGENET100_ROOT}" --out-dir /content/lsso_cv_runs \
  --mixer lsso --rank 16 \
  --image-size 224 --patch-size 8 --dim 256 --depth 8 --heads 8 \
  --epochs 120 --batch-size 256 --workers 16 --lr 1e-3 --warmup-epochs 10 \
  --weight-decay 0.05 --label-smoothing 0.1 --randaugment --mixup 0.8 --cutmix 1.0 \
  --drop-path 0.1 --amp bf16 --seed 1

{"event": "start", "torch": "2.11.0+cu128", "cuda": true, "gpu": "NVIDIA A100-SXM4-40GB", "args": {"dataset": "imagefolder", "data_root": "/content/data/imagenet100", "out_dir": "/content/lsso_cv_runs", "mixer": "lsso", "rank": 16, "image_size": 224, "patch_size": 8, "dim": 256, "depth": 8, "heads": 8, "mlp_ratio": 4, "dropout": 0, "drop_path": 0.1, "gamma_max": 0.3, "theta_gamma_init": -4, "epochs": 120, "batch_size": 256, "workers": 16, "lr": 0.001, "weight_decay": 0.05, "warmup_epochs": 10, "label_smoothing": 0.1, "mixup": 0.8, "cutmix": 1.0, "randaugment": true, "grad_clip": 1.0, "amp": "bf16", "seed": 1, "log_every": 50, "elastic_eval_ranks": "", "save_best": true}}
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation mig

In [ ]:
# ImageNet-100 LSSO-r32 patch=8 + elastic rank inference
!python /content/lsso_cv_colab.py \
  --dataset imagefolder --data-root "{IMAGENET100_ROOT}" --out-dir /content/lsso_cv_runs \
  --mixer lsso --rank 32 \
  --image-size 224 --patch-size 8 --dim 256 --depth 8 --heads 8 \
  --epochs 120 --batch-size 256 --workers 12 --lr 1e-3 --warmup-epochs 10 \
  --weight-decay 0.05 --label-smoothing 0.1 --randaugment --mixup 0 --cutmix 0 \
  --drop-path 0.1 --amp bf16 --seed 1 \
  --elastic-eval-ranks 32,24,16,8

{"event": "start", "torch": "2.11.0+cu128", "cuda": true, "gpu": "NVIDIA A100-SXM4-40GB", "args": {"dataset": "imagefolder", "data_root": "/content/data/imagenet100", "out_dir": "/content/lsso_cv_runs", "mixer": "lsso", "rank": 32, "image_size": 224, "patch_size": 8, "dim": 256, "depth": 8, "heads": 8, "mlp_ratio": 4, "dropout": 0, "drop_path": 0.1, "gamma_max": 0.3, "theta_gamma_init": -4, "epochs": 120, "batch_size": 256, "workers": 12, "lr": 0.001, "weight_decay": 0.05, "warmup_epochs": 10, "label_smoothing": 0.1, "mixup": 0.0, "cutmix": 0.0, "randaugment": true, "grad_clip": 1.0, "amp": "bf16", "seed": 1, "log_every": 50, "elastic_eval_ranks": "32,24,16,8", "save_best": true}}
{"event": "header", "run": "imagefolder_lsso_r32_d256_L8_h8_img224_p8_s1", "params": 6062052, "tokens": 784, "mixer_macs_est": 173932544, "classes": 100, "train_size": 130000, "val_size": 5000}
{"event": "epoch", "epoch": 1, "lr": 0.00010019723865877712, "train_loss": 4.384144607145171, "train_top1": 4.544193

## View results

In [ ]:
import pandas as pd
from pathlib import Path
p = Path('/content/lsso_cv_runs/summary.csv')
if p.exists(): display(pd.read_csv(p))
else: print('No summary yet.')

,dataset,mixer,rank,active_rank,dim,depth,heads,image_size,patch_size,tokens,...,mixer_macs_est,best_top1,final_top1,final_top5,final_loss,final_samples_per_sec,max_mem_gb,diag_gamma_over_mu,diag_correction_ratio,diag_effective_rank
0,imagefolder,lsso,32,32,256,8,8,224,8,784,...,173932544,78.3,78.30,92.86,1.528090,360.561793,19.642698,0.04863,0.845699,2.490515
1,imagefolder,lsso,32,32,256,8,8,224,8,784,...,173932544,78.3,78.30,92.86,1.528090,371.551244,19.642698,0.04863,0.845699,2.490515
2,imagefolder,lsso,32,24,256,8,8,224,8,784,...,154800128,78.3,76.92,91.92,1.579810,370.154245,19.642698,0.04863,0.835079,2.505709
3,imagefolder,lsso,32,16,256,8,8,224,8,784,...,136577024,78.3,70.96,89.10,1.793965,367.584689,19.642698,0.04863,0.816306,2.560011
4,imagefolder,lsso,32,8,256,8,8,224,8,784,...,119238656,78.3,46.90,74.34,3.060983,369.839214,19.642698,0.04863,0.772670,2.469090


In [ ]:
# Optional: copy results to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/lsso_cv_runs
# !cp -r /content/lsso_cv_runs/* /content/drive/MyDrive/lsso_cv_runs/

In [ ]:
!zip -r /content/lsso_cv_runs.zip /content/lsso_cv_runs

from google.colab import files
files.download('/content/lsso_cv_runs.zip')

  adding: content/lsso_cv_runs/ (stored 0%)
  adding: content/lsso_cv_runs/imagefolder_lsso_r32_d256_L8_h8_img224_p8_s1_best.pt (deflated 7%)
  adding: content/lsso_cv_runs/imagefolder_lsso_r32_d256_L8_h8_img224_p8_s1.jsonl (deflated 75%)
  adding: content/lsso_cv_runs/summary.csv (deflated 60%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
from google.colab import drive
from google.colab import runtime

# 1. 打包实验结果
!zip -q -r /content/lsso_cv_runs.zip /content/lsso_cv_runs
print("ZIP 压缩完成: /content/lsso_cv_runs.zip")

# 2. 挂载 Google Drive
drive.mount('/content/drive')

# 3. 复制到云盘
!mkdir -p /content/drive/MyDrive/lsso_cv_results
!cp /content/lsso_cv_runs.zip /content/drive/MyDrive/lsso_cv_results/
print("已成功复制到 Google Drive: /content/drive/MyDrive/lsso_cv_results/lsso_cv_runs.zip")

# 4. 断开并释放运行实例
print("正在关闭运行实例以节省计算单元...")
runtime.unassign()


zip error: Nothing to do! (try: zip -q -r /content/lsso_cv_runs.zip . -i /content/lsso_cv_runs)
ZIP 压缩完成: /content/lsso_cv_runs.zip


MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
import re

# 读取原始脚本
with open('/content/lsso_cv_colab.py', 'r') as f:
    code = f.read()

# Nystrom Attention 算子实现
nystrom_code = """
class NystromAttention(nn.Module):
    def __init__(self, dim, heads, num_landmarks=32):
        super().__init__()
        self.heads = heads; self.dh = dim // heads; self.num_landmarks = num_landmarks
        self.qkv = nn.Linear(dim, dim*3, bias=False); self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, D = x.shape; H = self.heads; dh = self.dh; m = self.num_landmarks
        qkv = self.qkv(x).reshape(B, N, 3, H, dh).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * (dh ** -0.5)
        stride = max(1, N // m)
        q_landmarks = q[:, :, ::stride, :][:, :, :m, :]
        k_landmarks = k[:, :, ::stride, :][:, :, :m, :]

        # 显式转换类型以兼容 AMP bf16
        kernel_1 = torch.nn.functional.softmax(torch.matmul(q, k_landmarks.transpose(-2, -1)), dim=-1).to(q.dtype)
        kernel_2 = torch.nn.functional.softmax(torch.matmul(q_landmarks, k_landmarks.transpose(-2, -1)), dim=-1).to(q.dtype)
        kernel_3 = torch.nn.functional.softmax(torch.matmul(q_landmarks, k.transpose(-2, -1)), dim=-1).to(q.dtype)
        kernel_2_inv = torch.linalg.pinv(kernel_2.float()).to(q.dtype)

        x = torch.matmul(torch.matmul(kernel_1, kernel_2_inv), torch.matmul(kernel_3, v))
        x = x.transpose(1, 2).reshape(B, N, D)
        return self.proj(x)
"""

# 注入算子并修改相关逻辑以支持 `--mixer nystrom`
if "class NystromAttention" not in code:
    code = code.replace("class LSSO(nn.Module):", nystrom_code + "\nclass LSSO(nn.Module):")
    code = code.replace("self.is_mha=(mixer=='mha')", "self.is_mha=(mixer=='mha'); self.is_nystrom=(mixer=='nystrom')")
    code = code.replace("if self.is_mha: self.mix=nn.MultiheadAttention(dim,heads,dropout=drop,batch_first=True)", "if self.is_mha: self.mix=nn.MultiheadAttention(dim,heads,dropout=drop,batch_first=True)\n        elif self.is_nystrom: self.mix=NystromAttention(dim,heads,rank)")
    code = code.replace("['mha','lsso','lsso-no-global']", "['mha','lsso','lsso-no-global','nystrom']")
    code = code.replace("if mixer=='mha': return 4*N*D*D + 2*N*N*D", "if mixer=='mha': return 4*N*D*D + 2*N*N*D\n    if mixer=='nystrom': return 4*N*D*D + 2*N*r*D + 2*N*r*r")

    with open('/content/lsso_cv_colab.py', 'w') as f:
        f.write(code)
    print("成功注入 Nystromformer 算子！")
else:
    print("Nystromformer 算子已存在。请重新运行 %%writefile 单元格恢复原始脚本，然后再运行本单元格。")


成功注入 Nystromformer 算子！


In [ ]:
# ImageNet-100 Nystromformer patch=8 (使用 rank=32 作为 landmarks 数量)
!python /content/lsso_cv_colab.py \
  --dataset imagefolder --data-root "/content/data/imagenet100" --out-dir /content/lsso_cv_runs \
  --mixer nystrom --rank 32 \
  --image-size 224 --patch-size 8 --dim 256 --depth 8 --heads 8 \
  --epochs 120 --batch-size 256 --workers 12 --lr 1e-3 --warmup-epochs 10 \
  --weight-decay 0.05 --label-smoothing 0.1 --randaugment --mixup 0.8 --cutmix 1.0 \
  --drop-path 0.1 --amp bf16 --seed 1

{"event": "start", "torch": "2.11.0+cu128", "cuda": true, "gpu": "NVIDIA A100-SXM4-40GB", "args": {"dataset": "imagefolder", "data_root": "/content/data/imagenet100", "out_dir": "/content/lsso_cv_runs", "mixer": "nystrom", "rank": 32, "image_size": 224, "patch_size": 8, "dim": 256, "depth": 8, "heads": 8, "mlp_ratio": 4, "dropout": 0, "drop_path": 0.1, "gamma_max": 0.3, "theta_gamma_init": -4, "epochs": 120, "batch_size": 256, "workers": 12, "lr": 0.001, "weight_decay": 0.05, "warmup_epochs": 10, "label_smoothing": 0.1, "mixup": 0.8, "cutmix": 1.0, "randaugment": true, "grad_clip": 1.0, "amp": "bf16", "seed": 1, "log_every": 50, "elastic_eval_ranks": "", "save_best": true}}
{"event": "header", "run": "imagefolder_nystrom_r32_d256_L8_h8_img224_p8_s1", "params": 6588260, "tokens": 784, "mixer_macs_est": 219971584, "classes": 100, "train_size": 130000, "val_size": 5000}
Traceback (most recent call last):
  File "/content/lsso_cv_colab.py", line 299, in <module>
    if __name__=='__main__'